# Bulk download VIIRS active fire detections monthly files 

In [ ]:
# required dependency
%pip install paramiko -q

In [ ]:
import fsspec 
import os 
from datetime import datetime
import gzip 
import shutil

In [ ]:
# connect to UMD SFTP server 
fs = fsspec.filesystem('sftp', host="fuoco.geog.umd.edu", username="fire", password="burnt")
noaa20 = fs.ls('/data/VIIRS/C2/VJ114IMGML/')
# use /data/VIIRS/C2/VNP14IMGML/ for SNPP
print(f"{len(noaa20)} monthly files available.\n{noaa20[0]} to {noaa20[-1]}")

outdir_string = "../data/FEDSinput/VIIRS/VJ114IMGML"
outdir = os.makedirs(outdir_string, exist_ok=True)

target_year = 2025
filtered = [f for f in noaa20 if datetime.strptime(f.split("/")[-1].split(".")[1], "%Y%m").year == target_year]
print(f"Downloading {len(filtered)} monthly files for {target_year}")

# Downloading 1 year took me just under 3 minutes, for reference
fs.get(filtered, outdir_string)

for filename in os.listdir(outdir_string):
    if filename.endswith('.csv.gz'):
        input_path = os.path.join(outdir_string, filename)
        output_filename = filename[:-3]  # remove the '.gz'
        output_path = os.path.join(outdir_string, output_filename)

        with gzip.open(input_path, 'rt') as f_in, open(output_path, 'wt') as f_out:
            shutil.copyfileobj(f_in, f_out)

        print(f"Extracted {filename} to {output_path}")
        os.remove(input_path)